# Step 3 — Train Gaussian Splatting

The COLMAP work is already done: steps 1 and 2 produced a folder holding
`images/` and `sparse/0/`. This notebook only trains.

**Before you run anything:** `Runtime` → `Change runtime type` → **T4 GPU**.

Upload that step-2 folder to Drive — as a `.zip` of the whole thing, a
`.tar.gz`, or the plain folder — then paste its path into **cell 6**, and pick
`SCAN_TYPE` / `PRESET` for the kind of scene you shot. Everything after that
finds `images/` and `sparse/0/` on its own and starts training.

Rough timings on a free-tier T4: setup 10 minutes, training 45-70 minutes at
1600px, roughly three times that at 3200px.

Every checkpoint is copied to Drive the moment it appears, so if Colab cuts the
session off you keep whatever finished.

## 1. GPU

In [ ]:
import subprocess

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No GPU attached. Runtime > Change runtime type > T4 GPU.")

print("GPU:", gpu.stdout.strip())

# CUDA compute capability, needed when compiling the rasteriser.
name = gpu.stdout.lower()
if "t4" in name:
    ARCH = "7.5"
elif "l4" in name:
    ARCH = "8.9"
elif "a100" in name:
    ARCH = "8.0"
else:
    ARCH = "7.5"
    print("Unknown card — assuming 7.5. If compilation fails, set ARCH by hand.")
print("ARCH =", ARCH)

## 2. Google Drive

In [ ]:
import os

if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
print("Drive mounted.")

## 3. Cấu hình — sửa một dòng ở đây

**Configuration — the only cell you edit.** Dán đường dẫn kết quả bước 2 vào
`COLMAP_INPUT`, chọn preset cho đúng loại vật thể, chạy hết. Tên dự án lấy theo
tên file, nên `Mèo.zip` ghi ra `Meo_30000.ply`.

Hai preset là hai bộ tham số đã chốt, không phải gợi ý để nghịch:

| | FLAT_OBJECT | COMPLEX_OBJECT |
|---|---|---|
| dùng khi | vật thể có chữ nhỏ cần đọc được | vật thể nhiều gờ cạnh |
| số ảnh | 150–180 | 180–220 |
| chặng 0 thu nhỏ về | **3200 px** | 2400 px |
| `densify_grad_threshold` | 0.00015 | 0.0002 |
| `percent_dense` | **0.005** | 0.01 |
| `densify_until_iter` | 20000 | 15000 |
| `scaling_lr` | 0.002 | 0.005 |
| `sh_degree` | 3 | 3 |
| trần số hạt | 2.500.000 | 2.500.000 |

Hai dòng ngược trực giác, nói rõ cho khỏi sửa nhầm:

- `percent_dense` là ngưỡng quyết định **nhân bản** (clone) hay **tách**
  (split) một hạt. Hạ 0.01 → 0.005 đẩy nhiều hạt sang nhóm "tách", tức là
  sinh hạt nhỏ hơn trên mặt phẳng. Đây là đòn bẩy trực tiếp nhất cho chữ nhỏ.
- `scaling_lr` 0.005 → 0.002 kìm hạt bị kéo dãn mà không phải đụng vào loss.

Biến `LIMIT_VRAM` của bản cũ đã bị bỏ hẳn — xem lời giải thích trong ô dưới.

`MILESTONE` chọn một trong bốn mốc. Mỗi mốc khác mốc ngay trước nó **đúng một
thứ**, vì đó là cách duy nhất biết được thứ nào ăn tiền:

| mốc | bộ vẽ | cờ mới |
|---|---|---|
| `V1_5` | `dr_aa` | không — và không dùng bảng preset ở trên |
| `V2a` | `3dgs_accel` | không |
| `V2b` | `3dgs_accel` | `--antialiasing` |
| `V2c` | `3dgs_accel` | `--antialiasing --optimizer_type sparse_adam` |

Mốc `V2a` trông thừa nhưng không thừa. `train.py` truyền
`separate_sh=SPARSE_ADAM_AVAILABLE`, nghĩa là chỉ cần **cài được** bộ vẽ
`3dgs_accel` là đường tính SH đã đổi, chưa bật cờ nào cả. Không có `V2a` thì
`V1_5 → V2b` gộp hai thay đổi vào một con số và không tách ra được nữa.

Tên dự án tự đeo hậu tố mốc, nên bốn lần chạy không đè file của nhau.

> Mốc `V2` của bản cũ đã tách làm ba. Gõ `"V2"` sẽ báo lỗi kèm giải thích chứ
> không chạy im lặng — số cũ đo trên `dr_aa`, không so thẳng với `V2a` được.

> **Quét cả căn phòng đang bị khoá.** Preset `ENTIRE_ROOM` vẫn nằm nguyên trong
> mã nguồn và đã chạy được ở chặng 3, nhưng chặng 1 và chặng 2 chưa xong nên
> chọn nó sẽ báo lỗi ngay thay vì để bạn chạy năm tiếng rồi nhận về rác. Còn
> thiếu gì và mở lại thế nào: [`docs/ROOM_MODE.md`](../docs/ROOM_MODE.md).

In [ ]:
# ============ SỬA MẤY DÒNG NÀY / EDIT THESE LINES ============
#
#  ┌───────────────────────────────────────────────────────────────────┐
#  │  Dán đường dẫn tới kết quả bước 2 trên Drive vào giữa hai dấu " " │
#  │  Paste the path to your step-2 result on Drive between the quotes │
#  └───────────────────────────────────────────────────────────────────┘
#
#  Lấy đường dẫn: bấm biểu tượng thư mục ở thanh bên trái Colab, mở
#  drive/MyDrive, tìm file, bấm ba chấm, "Copy path", dán vào đây.
#
#  Nhận cả ba dạng — all three shapes work:
#       Mèo.zip          zip của cả thư mục Mèo_3d          (thường dùng nhất)
#       Mèo_3d.tar.gz
#       Mèo_3d           thư mục để thẳng trên Drive

COLMAP_INPUT = "/content/drive/MyDrive/img3dpl/CHANGE_ME.zip"

# ---- Loại vật thể: dòng này quyết định toàn bộ phần còn lại ---------------
#
#   "FLAT_OBJECT"     vật thể có mặt phẳng chữ nhỏ cần đọc được — trang sách,
#                     bìa truyện, hộp bài, nhãn hàng. Ảnh chụp ở 3200px.
#   "COMPLEX_OBJECT"  vật thể nhiều gờ cạnh, ít chữ. Ảnh 2400px.
#
PRESET = "FLAT_OBJECT"      # "FLAT_OBJECT" | "COMPLEX_OBJECT"

# ---- Mốc thử nghiệm -------------------------------------------------------
#
#  Đừng đổi hai thứ cùng lúc rồi ngồi đoán cái nào ăn tiền.
#
#  Bản Inria tháng 10/2024 mang theo hai cờ đáng bật, nhưng cờ thứ hai kéo theo
#  việc đổi cả bộ vẽ — và đổi bộ vẽ thì TỰ NÓ đã đổi đường render rồi. train.py
#  truyền separate_sh=SPARSE_ADAM_AVAILABLE, tức là chỉ cần CÀI ĐƯỢC nhánh
#  3dgs_accel là đường tính SH đã khác, chưa bật cờ nào cả. Nên 3dgs_accel
#  KHÔNG phải nền trung tính, và phải có một mốc đo riêng cho mỗi bước.
#
#   "V1_5"  bộ vẽ dr_aa, không cờ mới, không bảng preset. Chỉ đẩy ảnh xuống
#           RAM và bỏ LIMIT_VRAM. CHẠY MỐC NÀY TRƯỚC — rất có thể nó đã đủ.
#   "V2a"   bộ vẽ 3dgs_accel + bảng preset + các bản vá. Không cờ mới. Đây là
#           NỀN để so V2b và V2c; thiếu nó thì không tách được công của bộ vẽ
#           ra khỏi công của cờ.
#   "V2b"   V2a + --antialiasing
#   "V2c"   V2b + --optimizer_type sparse_adam
#
MILESTONE = "V2c"           # "V1_5" | "V2a" | "V2b" | "V2c"

# True = 3DGS giữ lại 1/8 số ảnh làm bộ đối chứng, để ô 9 đo được PSNR/SSIM.
# Bật khi đang so V1_5 với V2. Tắt cho lần chạy cuối, để mô hình dùng hết ảnh.
EVAL = True

# Tên ngắn cho lần quét này, dùng đặt tên file .ply đầu ra.
# Để trống = tự lấy theo tên file ở trên ("Mèo.zip" -> "Meo").
PROJECT = ""

# Thư mục trên Drive chứa file .ply hoàn chỉnh. Tự tạo nếu chưa có.
OUTPUT_DIR = "/content/drive/MyDrive/img3dpl/results"

# ================= HẾT PHẦN CẦN SỬA / NOTHING TO EDIT BELOW =================
#
# Biến LIMIT_VRAM của bản cũ đã bị bỏ hẳn. Nó nhân đôi densify_grad_threshold
# và cắt densification sớm 3000 iter, nên vùng chữ nhỏ không bao giờ được tách
# hạt — từ iter 12000 tới 30000 optimizer chỉ còn cách kéo dãn hạt sẵn có, ra
# đúng những vệt dài trên mặt phẳng. Nó tồn tại chỉ vì lệnh train cũ thiếu
# --data_device cpu nên 320 ảnh float32 nằm chình ình 7,4 GB trên VRAM.
# Đẩy ảnh xuống RAM là hết lý do tồn tại của nó.

import copy
import os
import re
import unicodedata

# Bảng ba preset. Ba nhóm khoá:
#   cli      -> truyền thẳng vào train.py
#   patch    -> hằng số ghi vào mã nguồn (không có đường CLI nào tới được)
#   pipeline -> độ phân giải, số ảnh, hậu xử lý — dùng cho các ô sau
PRESETS = {
    "FLAT_OBJECT": {
        "cli": {
            "--iterations": 30000,
            "--densify_from_iter": 500,
            "--densify_until_iter": 20000,
            "--densification_interval": 100,
            # Hạ từ 0.0002 xuống. Ngược hẳn với LIMIT_VRAM cũ (0.0004).
            "--densify_grad_threshold": 0.00015,
            # Đòn bẩy trực tiếp nhất cho chữ nhỏ: percent_dense là ngưỡng quyết
            # định nhân bản (clone) hay tách (split). Hạ 0.01 -> 0.005 đẩy nhiều
            # hạt sang nhóm "tách", tức là sinh hạt NHỎ HƠN trên mặt phẳng.
            "--percent_dense": 0.005,
            # Giữ nguyên 3000. Reset opacity là cơ chế diệt hạt ma có sẵn.
            "--opacity_reset_interval": 3000,
            # Cách rẻ nhất kìm hạt bị kéo dãn, không phải đụng vào loss.
            "--scaling_lr": 0.002,
            "--sh_degree": 3,
            "--lambda_dssim": 0.2,
        },
        "patch": {
            "MAX_GAUSSIANS": 2_500_000,
            "MIN_OPACITY": 0.005,
            "ANISO_LAMBDA": 0.01,
            "ANISO_MAX_RATIO": 4.0,
        },
        "pipeline": {
            "resize_px": 3200,
            "so_anh": (150, 180),
            "saves": [7000, 15000, 25000, 30000],
            "compress_sh_degree": 1,
            "masking": True,
            "floater": "nhe",
            "opacity_loc": 0.05,
            "scale_percentile": None,
        },
    },
    "COMPLEX_OBJECT": {
        "cli": {
            "--iterations": 30000,
            "--densify_from_iter": 500,
            "--densify_until_iter": 15000,
            "--densification_interval": 100,
            "--densify_grad_threshold": 0.0002,
            "--percent_dense": 0.01,
            "--opacity_reset_interval": 3000,
            "--scaling_lr": 0.005,
            "--sh_degree": 3,
            "--lambda_dssim": 0.2,
        },
        "patch": {
            "MAX_GAUSSIANS": 2_500_000,
            "MIN_OPACITY": 0.005,
            "ANISO_LAMBDA": 0.0,
            "ANISO_MAX_RATIO": 4.0,
        },
        "pipeline": {
            "resize_px": 2400,
            "so_anh": (180, 220),
            "saves": [7000, 15000, 25000, 30000],
            "compress_sh_degree": 1,
            "masking": True,
            "floater": "nhe",
            "opacity_loc": 0.05,
            "scale_percentile": None,
        },
    },
    "ENTIRE_ROOM": {
        "cli": {
            "--iterations": 30000,
            "--densify_from_iter": 500,
            "--densify_until_iter": 15000,
            "--densification_interval": 100,
            "--densify_grad_threshold": 0.0004,
            "--percent_dense": 0.01,
            "--opacity_reset_interval": 3000,
            "--scaling_lr": 0.005,
            # 24 hệ số f_rest thay vì 45 -> mỗi hạt nhẹ đi khoảng 35% bộ nhớ,
            # cùng ngần ấy VRAM chứa được nhiều hạt hơn. Phòng cần phủ rộng
            # hơn là cần phản chiếu đẹp.
            "--sh_degree": 2,
            "--lambda_dssim": 0.2,
        },
        "patch": {
            "MAX_GAUSSIANS": 2_000_000,
            "MIN_OPACITY": 0.01,
            "ANISO_LAMBDA": 0.0,
            "ANISO_MAX_RATIO": 4.0,
        },
        "pipeline": {
            "resize_px": 1600,
            "so_anh": (300, 400),
            "saves": [7000, 15000, 25000, 30000],
            "compress_sh_degree": 0,
            "masking": False,
            "floater": "manh",
            "opacity_loc": 0.02,
            "scale_percentile": 99.5,
        },
    },
}

# Bảng mốc. Mỗi mốc khác mốc ngay trước nó ĐÚNG một thứ — đó là toàn bộ lý do
# bảng này tồn tại. Bốn khoá:
#   rasterizer  nhánh của submodule diff-gaussian-rasterization phải build.
#               "dr_aa"      nhánh .gitmodules ghim sẵn. CÓ antialiasing,
#                            KHÔNG có SparseGaussianAdam.
#               "3dgs_accel" nhánh README của Inria bảo checkout. Có CẢ hai,
#                            nên nó là tập cha chứ không phải một đánh đổi.
#   co_bat      cờ dạng bật/tắt (argparse store_true). Không có giá trị đi kèm
#               nên không nhét vào dict cli được — nhét vào là ô 8 sinh ra
#               "--antialiasing True" và argparse chết vì thừa tham số.
#   sparse_adam đặt --optimizer_type. Tách riêng vì nó là thứ duy nhất bắt
#               buộc phải có 3dgs_accel: train.py sys.exit ngay nếu thiếu.
#   day_du      True  = dùng cả bảng preset + các bản vá mã nguồn.
#               False = đúng hai thay đổi một dòng, còn lại để 3DGS tự lo.
MOC = {
    "V1_5": {"rasterizer": "dr_aa",
             "co_bat": [], "sparse_adam": False, "day_du": False},
    "V2a":  {"rasterizer": "3dgs_accel",
             "co_bat": [], "sparse_adam": False, "day_du": True},
    "V2b":  {"rasterizer": "3dgs_accel",
             "co_bat": ["--antialiasing"], "sparse_adam": False, "day_du": True},
    "V2c":  {"rasterizer": "3dgs_accel",
             "co_bat": ["--antialiasing"], "sparse_adam": True, "day_du": True},
}

# Cờ nào render.py cũng phải nhận lại. render.py chỉ dựng ModelParams và
# PipelineParams, nên chỉ cờ thuộc PipelineParams mới truyền cho nó được —
# đưa --optimizer_type (OptimizationParams) sang là argparse báo lỗi.
# Vì sao phải truyền lại: xem chú thích dài ở ô 9.
CO_RENDER_HOP_LE = {"--antialiasing"}

# Mốc "V2" của bản cũ đã tách làm ba. Không nhận im lặng, vì "V2" cũ chạy trên
# dr_aa còn V2a chạy trên 3dgs_accel — hai con số đó không so thẳng với nhau
# được, và để nó chạy tiếp là làm hỏng chính phép đo mà bảng này sinh ra để đo.
MOC_DA_BO = {
    "V2": "Mốc 'V2' cũ đã tách làm ba: V2a, V2b, V2c.\n"
          "  V2a  = 'V2' cũ nhưng trên bộ vẽ 3dgs_accel (đường render đã khác)\n"
          "  V2b  = V2a + --antialiasing\n"
          "  V2c  = V2b + --optimizer_type sparse_adam\n"
          "Số PSNR/SSIM đo bằng 'V2' cũ không so thẳng với V2a được — bộ vẽ\n"
          "khác nhau. Muốn so thì đo lại V2a.",
}
HOP_LE = {"OBJECT": ("FLAT_OBJECT", "COMPLEX_OBJECT"),
          "ROOM": ("ENTIRE_ROOM",)}

# Loại quét suy ra từ preset, không bắt người dùng khai hai lần rồi khai lệch
# nhau. Bảng này là chiều ngược của HOP_LE.
LOAI_QUET = {ten: loai for loai, ds in HOP_LE.items() for ten in ds}

# ---------------------------------------------------------------------------
# CỔNG CHẶN CHẾ ĐỘ QUÉT PHÒNG
#
# Toàn bộ code ROOM bên dưới đã viết xong và đã chạy thử ở Chặng 3 — preset
# ENTIRE_ROOM, sh_degree 2, trần 2 triệu hạt, lọc theo phân vị scale. Không có
# dòng nào bị xoá. Nhưng hai chặng trước nó thì chưa xong, và một quy trình
# nửa vời còn tệ hơn không có: người dùng chạy hết năm tiếng rồi nhận về rác.
#
# Bỏ cổng này khi ba việc trong docs/ROOM_MODE.md đã làm xong.
# ---------------------------------------------------------------------------
CHAN_ROOM = True

THIEU_GI_CHO_ROOM = """\
Chế độ quét phòng (PRESET = "ENTIRE_ROOM") đang bị khoá, chưa dùng được.

Chặng 3 thì xong rồi, nhưng hai chặng trước nó thì chưa:

  1. Chặng 1 vẫn dùng exhaustive_matcher. Đi bộ trong phòng thì ảnh ở hai
     đầu phòng không nhìn thấy gì chung, nên phần lớn trong 124.750 cặp
     (500 ảnh) trả về rỗng mà vẫn tốn đủ thời gian tính. Sai công cụ.

  2. Chặng 2 vì thế nghẹn 3-5 tiếng trên i3 hai nhân, và chưa ai đo thật.

  3. Dọn hạt ma cho phòng mới có mỗi dòng chữ nhắc ở ô 10, chưa có code.
     Lọc opacity với phân vị scale là chưa đủ cho một căn phòng.

Chạy tiếp bây giờ là mất cả buổi để nhận về một mô hình không dùng được.

Đọc docs/ROOM_MODE.md để biết còn thiếu đúng những gì và mở lại thế nào.

Quét vật thể thì vẫn chạy bình thường: PRESET = "FLAT_OBJECT" hoặc
"COMPLEX_OBJECT"."""


def get_training_config(scan_type, preset, milestone):
    """Trả về dict cấu hình đã kiểm tra, hoặc dừng hẳn nếu tổ hợp vô lý."""
    if scan_type not in HOP_LE:
        raise SystemExit(f"SCAN_TYPE phải là OBJECT hoặc ROOM, không phải {scan_type!r}")
    if preset not in PRESETS:
        # Chỉ liệt kê những preset đang thật sự dùng được, không kể cái đang khoá.
        dung_duoc = [t for t in PRESETS
                     if not (CHAN_ROOM and LOAI_QUET[t] == "ROOM")]
        raise SystemExit(f"PRESET phải là một trong {dung_duoc}, không phải {preset!r}")
    if preset not in HOP_LE[scan_type]:
        raise SystemExit(
            f"SCAN_TYPE={scan_type} không đi được với PRESET={preset}.\n"
            f"  OBJECT -> FLAT_OBJECT hoặc COMPLEX_OBJECT\n"
            f"  ROOM   -> ENTIRE_ROOM")
    if milestone in MOC_DA_BO:
        raise SystemExit(MOC_DA_BO[milestone])
    if milestone not in MOC:
        raise SystemExit(f"MILESTONE phải là một trong {list(MOC)}, "
                         f"không phải {milestone!r}")

    # Cổng chặn đứng ở đây, sau phần kiểm tra tổ hợp — để lỡ ai gõ sai tên
    # preset thì vẫn nhận đúng lời báo lỗi cũ, chứ không bị lời báo về phòng.
    if CHAN_ROOM and scan_type == "ROOM":
        raise SystemExit(THIEU_GI_CHO_ROOM)

    cfg = copy.deepcopy(PRESETS[preset])

    if scan_type == "ROOM" and cfg["pipeline"]["masking"]:
        raise SystemExit("Chế độ ROOM không được bật masking 2D — nền chính là "
                         "tường, mask nó đi là mất luôn cảnh.")

    # [ĐÃ CHỐT] Hai cờ này luôn có mặt, ở mọi preset, mọi mốc.
    #   --data_device cpu : ảnh nằm ở RAM chứ không phải VRAM. Giải phóng ~7 GB.
    #   -r 1              : mặc định của 3DGS là -1, nghĩa là nó TỰ HẠ mọi ảnh
    #                       rộng hơn 1600px xuống 1600px và chỉ in một dòng
    #                       cảnh báo nhỏ. Quên cờ này là công của chặng 0 bị
    #                       vứt đi trong im lặng.
    cfg["cli"]["--data_device"] = "cpu"
    cfg["cli"]["-r"] = 1

    moc = MOC[milestone]

    if not moc["day_du"]:
        # Mốc A: đúng hai thay đổi một dòng, còn lại để 3DGS tự lo. Giữ lại
        # trần số hạt vì không có nó thì ngưỡng mặc định 0.0002 có thể ăn hết
        # 15 GB VRAM trước iter 15000.
        giu = {"--iterations", "--data_device", "-r"}
        cfg["cli"] = {k: v for k, v in cfg["cli"].items() if k in giu}
        cfg["patch"]["ANISO_LAMBDA"] = 0.0
        cfg["patch"]["MIN_OPACITY"] = 0.005      # đúng hằng số gốc trong train.py
        cfg["va"] = ["cap", "log"]
    else:
        cfg["va"] = ["uint8", "cap", "log"]
        if cfg["patch"]["ANISO_LAMBDA"] > 0:
            cfg["va"].append("aniso")

    # Đặt SAU phần cắt cli của V1_5, không thì mốc nào cắt cli cũng nuốt luôn
    # cờ vừa thêm mà không ai thấy.
    cfg["rasterizer"] = moc["rasterizer"]
    cfg["co_bat"] = list(moc["co_bat"])
    if moc["sparse_adam"]:
        cfg["cli"]["--optimizer_type"] = "sparse_adam"
    cfg["co_render"] = [c for c in cfg["co_bat"] if c in CO_RENDER_HOP_LE]

    return cfg


if "CHANGE_ME" in COLMAP_INPUT:
    raise SystemExit("Sửa COLMAP_INPUT ở trên thành đường dẫn của chính bạn.")

if not os.path.exists(COLMAP_INPUT):
    raise SystemExit(f"Không thấy trên Drive / not found: {COLMAP_INPUT}")

# Preset quyết định luôn loại quét — người dùng chỉ phải gõ một dòng.
SCAN_TYPE = LOAI_QUET.get(PRESET, "OBJECT")

CFG = get_training_config(SCAN_TYPE, PRESET, MILESTONE)


def short_name(path):
    """Tên dự án lấy từ tên file: "Mèo.zip" -> "Meo".

    Bỏ dấu tiếng Việt và mọi ký tự lạ, vì tên này đi vào tên file .ply và
    vào dòng lệnh — tên có dấu hay khoảng trắng là nguồn lỗi khó đoán.
    """
    name = os.path.basename(path.rstrip("/"))
    for suffix in (".tar.gz", ".tgz", ".tar", ".zip"):
        if name.lower().endswith(suffix):
            name = name[: -len(suffix)]
            break
    name = name.replace("đ", "d").replace("Đ", "D")
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    name = re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")
    return name or "scan"


PROJECT = PROJECT.strip() or short_name(COLMAP_INPUT)
# Bốn mốc thì mốc nào cũng đeo hậu tố, không chừa ngoại lệ nào. Có ngoại lệ là
# có ngày hai lần chạy đè file của nhau, mà nhìn tên file lại không biết đường.
PROJECT = f"{PROJECT}_{MILESTONE}"

SAVES = sorted(set(CFG["pipeline"]["saves"]))
ITERATIONS = max(SAVES + [int(CFG["cli"]["--iterations"])])
CFG["cli"]["--iterations"] = ITERATIONS
COMPRESS_SH_DEGREE = CFG["pipeline"]["compress_sh_degree"]
WORK = f"/content/{PROJECT}"

sh_min, sh_max = CFG["pipeline"]["so_anh"]
print("Đầu vào / input :", COLMAP_INPUT)
print("Tên dự án       :", PROJECT)
print("Đầu ra / output :", OUTPUT_DIR)
print("Loại cảnh       :", SCAN_TYPE, "/", PRESET, " · mốc", MILESTONE,
      "· eval" if EVAL else "· không eval")
print("Bộ vẽ           :", CFG["rasterizer"],
      "(ô cài đặt build nhánh này; hỏng thì tự lùi về dr_aa)")
print(f"Bộ ảnh nên là   : {sh_min}–{sh_max} tấm, cạnh dài "
      f"{CFG['pipeline']['resize_px']}px (làm ở chặng 0)")
print("Checkpoints     :", SAVES)

print("\nTham số truyền vào train.py:")
for co, gt in CFG["cli"].items():
    print(f"   {co} {gt}")
for co in CFG["co_bat"]:
    print(f"   {co}")
print("\nHằng số vá thẳng vào mã nguồn:")
for ten in CFG["va"]:
    if ten == "cap":
        print(f"   MAX_GAUSSIANS   {CFG['patch']['MAX_GAUSSIANS']:,}")
        print(f"   MIN_OPACITY     {CFG['patch']['MIN_OPACITY']}")
    elif ten == "aniso":
        print(f"   ANISO_LAMBDA    {CFG['patch']['ANISO_LAMBDA']}")
        print(f"   ANISO_MAX_RATIO {CFG['patch']['ANISO_MAX_RATIO']}")
print("Bản vá sẽ áp   :", ", ".join(CFG["va"]))

## 4. Find the scene and load it

Runs on its own — nothing to edit here.

Step 2 leaves a folder like this, and you can hand the whole thing over as a
`.zip`, a `.tar.gz`, or a plain folder:

```
Mèo_3d/
├── images/                 ← used: undistorted photos
├── sparse/0/*.bin          ← used: cameras, poses, sparse cloud
├── distorted/database.db   ← skipped, and it is the biggest file by far
├── distorted/sparse/0/     ← skipped: the model before undistortion
├── stereo/                 ← skipped: empty scaffolding for dense
└── run-colmap-*.sh         ← skipped
```

Only `images/` and `sparse/0/` are read out of the archive; everything else is
left inside it. That saves several gigabytes of Colab disk and a lot of waiting,
because `database.db` alone is usually larger than the rest put together.

The scene is located by looking for a folder that holds **both** `images/` and
`sparse/0/`, at any depth — so an extra wrapper folder inside the zip is fine,
and `distorted/sparse/0/` is never mistaken for the real one.

In [ ]:
import os
import shutil
import struct
import tarfile
import time
import zipfile

KEEP = ("images", "sparse")     # the only two folders train.py ever reads

shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK, exist_ok=True)
started = time.time()


def find_scene(paths):
    """The folder holding both images/ and sparse/0/. "" means the top level.

    Step 2 writes a second model under distorted/sparse/0 — the one from before
    undistortion, whose poses do not line up with the photos in images/. Asking
    for images/ as a sibling is what rules it out: only the real scene has one.
    """
    with_sparse, with_images = set(), set()
    for path in paths:
        parts = path.strip("/").split("/")
        for i, part in enumerate(parts[:-1]):
            prefix = "/".join(parts[:i])
            if part == "sparse" and parts[i + 1] == "0":
                with_sparse.add(prefix)
            elif part == "images":
                with_images.add(prefix)
    both = with_sparse & with_images
    return min(both, key=len) if both else None


def inside_scene(path, scene):
    """Path rewritten relative to the scene, or None if training does not want it."""
    parts = path.strip("/").split("/")
    prefix = scene.split("/") if scene else []
    if parts[:len(prefix)] != prefix:
        return None
    rest = parts[len(prefix):]
    if len(rest) < 2 or rest[0] not in KEEP:
        return None
    return "/".join(rest)


def give_up(paths):
    """No scene found — show what the input actually holds, then stop."""
    print("\nKhông tìm thấy thư mục nào chứa cả images/ và sparse/0/.")
    print("No folder inside holds both images/ and sparse/0/. What is in there:\n")
    folders = {}
    for path in paths:
        folders.setdefault(os.path.dirname(path), []).append(os.path.basename(path))
    for folder in sorted(folders)[:25]:
        names = sorted(folders[folder])
        print(f"  {folder or '.'}/  ({len(names)} files)  {', '.join(names[:4])}")
    raise SystemExit("Sai file? Cần kết quả của bước 2, không phải thư mục ảnh gốc.")


def write(fileobj, relative):
    """Save one file from the archive into WORK, keeping images/ and sparse/."""
    target = os.path.join(WORK, relative)
    os.makedirs(os.path.dirname(target), exist_ok=True)
    with open(target, "wb") as out:
        shutil.copyfileobj(fileobj, out)


def size_text(count):
    """Bytes as GB once GB is worth saying, otherwise MB."""
    return f"{count / 1e9:.2f} GB" if count >= 1e9 else f"{count / 1e6:.0f} MB"


taken = skipped = 0          # file counts
taken_size = skipped_size = 0

if os.path.isdir(COLMAP_INPUT):
    print("Đầu vào là thư mục — đang chép sang đĩa của Colab...")
    # Copy rather than train straight off Drive: training reads these files
    # thousands of times, and Drive is far slower than local disk.
    listing = []
    for folder, _, files in os.walk(COLMAP_INPUT):
        rel = os.path.relpath(folder, COLMAP_INPUT)
        rel = "" if rel == "." else rel
        listing += [os.path.join(rel, name) for name in files]

    scene = find_scene(listing)
    if scene is None:
        give_up(listing)

    for path in listing:
        source = os.path.join(COLMAP_INPUT, path)
        size = os.path.getsize(source)
        relative = inside_scene(path, scene)
        if relative is None:
            skipped, skipped_size = skipped + 1, skipped_size + size
            continue
        with open(source, "rb") as handle:
            write(handle, relative)
        taken, taken_size = taken + 1, taken_size + size

elif COLMAP_INPUT.lower().endswith(".zip"):
    print("Đầu vào là .zip — đang giải nén phần cần thiết...")
    with zipfile.ZipFile(COLMAP_INPUT) as archive:
        entries = [e for e in archive.infolist() if not e.is_dir()]
        scene = find_scene([e.filename for e in entries])
        if scene is None:
            give_up([e.filename for e in entries])

        for entry in entries:
            relative = inside_scene(entry.filename, scene)
            if relative is None:
                skipped, skipped_size = skipped + 1, skipped_size + entry.file_size
                continue
            with archive.open(entry) as handle:
                write(handle, relative)
            taken, taken_size = taken + 1, taken_size + entry.file_size

else:
    print("Đầu vào là .tar.gz — đang giải nén phần cần thiết...")
    with tarfile.open(COLMAP_INPUT) as archive:
        members = [m for m in archive.getmembers() if m.isfile()]
        scene = find_scene([m.name for m in members])
        if scene is None:
            give_up([m.name for m in members])

        for member in members:
            relative = inside_scene(member.name, scene)
            if relative is None:
                skipped, skipped_size = skipped + 1, skipped_size + member.size
                continue
            write(archive.extractfile(member), relative)
            taken, taken_size = taken + 1, taken_size + member.size

print(f"Thư mục cảnh / scene folder: {scene or '(gốc / top level)'}")
print(f"Đã lấy  : {taken} file, {size_text(taken_size)}")
print(f"Bỏ qua  : {skipped} file, {size_text(skipped_size)} "
      f"(distorted/, stereo/, script — training không đọc tới)")
print(f"Xong sau {time.time() - started:.0f}s")

# ---- Check that what came out is actually trainable -------------------------
sparse = f"{WORK}/sparse/0"
photos = sorted(f for f in os.listdir(f"{WORK}/images")
                if f.lower().endswith((".jpg", ".jpeg", ".png")))

if not photos:
    raise SystemExit("images/ rỗng — kiểm tra lại file đã tải lên Drive.")

for stem in ("cameras", "images", "points3D"):
    if not (os.path.exists(f"{sparse}/{stem}.bin")
            or os.path.exists(f"{sparse}/{stem}.txt")):
        raise SystemExit(f"Thiếu sparse/0/{stem}.bin — bước 2 chưa chạy xong?")

print(f"\nẢnh / images : {len(photos)}")
print("sparse/0     :", ", ".join(sorted(os.listdir(sparse))))

# images.bin starts with a uint64: how many photos COLMAP actually placed.
# A model built from 320 photos that only registered 90 of them trains fine and
# then looks wrong, so it is worth saying out loud before an hour of training.
if os.path.exists(f"{sparse}/images.bin"):
    with open(f"{sparse}/images.bin", "rb") as handle:
        registered = struct.unpack("<Q", handle.read(8))[0]
    print(f"Đã định vị   : {registered}/{len(photos)} ảnh")
    if registered < 0.7 * len(photos):
        print("!! Nhiều ảnh không được định vị. Model sẽ thiếu mảng.")
        print("!! Ảnh cạnh nhau cần trùng nhau khoảng 60-80%.")

## 5. Kiểm tra trước khi train

Hai việc, cả hai chạy trong vài giây và cả hai đều đáng làm trước khi tiêu một
tiếng đồng hồ.

**ĐO A — vị trí máy ảnh có đủ chính xác không.** Đọc sai số tái chiếu COLMAP đã
ghi sẵn trong `sparse/0/points3D.bin`. 3DGS khớp theo cường độ sáng, nên máy ảnh
lệch nửa pixel là cùng một nét chữ rơi vào hai chỗ khác nhau giữa hai góc nhìn,
và cách duy nhất optimizer dung hoà được là làm mờ nó đi.

Đây là nguồn mờ **hoàn toàn độc lập với densification**. Không tham số train nào
sửa được, và đổi sang AbsGS hay Improved-GS cũng không. Với cảnh gần phẳng —
trang sách, tờ giấy — đây là chỗ đáng nghi trước tiên, vì mọi điểm nằm trên một
mặt phẳng thì có vô số cách đặt máy ảnh cho ra cùng một bức ảnh.

**Ngân sách RAM và VRAM.** RAM hệ thống mới là nút thắt thật, không phải VRAM:
Colab free có ~12,7 GB và bộ ảnh nằm hết trong đó. Vượt trần thì ô này **dừng
lại** và nói phải bớt bao nhiêu tấm — thà biết ngay bây giờ còn hơn chết vì hết
bộ nhớ ở phút thứ bốn mươi.

In [ ]:
import struct

import numpy as np
from PIL import Image

# ===========================================================================
# ĐO A — chất lượng vị trí máy ảnh do chặng 2 dựng ra
#
# Đọc sai số tái chiếu COLMAP ghi sẵn trong points3D.bin. Đây là con số duy
# nhất nói được pose có đủ chính xác hay không, và nó đọc được NGAY BÂY GIỜ,
# trước khi tốn một tiếng train.
#
# Vì sao nó quan trọng riêng với chữ nhỏ: 3DGS khớp theo cường độ sáng. Máy
# ảnh lệch nửa pixel thì cùng một nét chữ rơi vào hai chỗ khác nhau giữa hai
# góc nhìn, và cách duy nhất optimizer dung hoà được là làm mờ nó đi. Đây là
# nguồn mờ hoàn toàn độc lập với densification — không tham số train nào sửa
# được, và đổi thuật toán cũng không.
# ===========================================================================


def doc_points3d(duong):
    """
    Đọc points3D.bin của COLMAP, trả về (sai số tái chiếu, độ dài track).

    Bố cục mỗi điểm: id (uint64), xyz (3 double), rgb (3 uint8),
    error (double), track_length (uint64), rồi track_length cặp uint32.
    """
    with open(duong, "rb") as f:
        kho = f.read()
    (so_diem,) = struct.unpack_from("<Q", kho, 0)
    vi_tri = 8
    sai_so = np.empty(so_diem)
    track = np.empty(so_diem, dtype=np.int64)
    for i in range(so_diem):
        # 8 id + 24 xyz + 3 rgb = 35 byte trước error
        (loi,) = struct.unpack_from("<d", kho, vi_tri + 35)
        (dai,) = struct.unpack_from("<Q", kho, vi_tri + 43)
        sai_so[i], track[i] = loi, dai
        vi_tri += 51 + dai * 8
    return sai_so, track


tep_diem = f"{WORK}/sparse/0/points3D.bin"
if os.path.exists(tep_diem):
    sai_so, track = doc_points3d(tep_diem)
    print(f"Điểm thưa    : {len(sai_so):,} điểm")
    print(f"Sai số tái chiếu : trung bình {sai_so.mean():.3f} px, "
          f"trung vị {np.median(sai_so):.3f} px, "
          f"phân vị 90 {np.percentile(sai_so, 90):.3f} px")
    print(f"Mỗi điểm được  : {track.mean():.1f} camera nhìn thấy (trung bình), "
          f"ít nhất {track.min()}")

    # Ngưỡng dưới đây là kinh nghiệm, không phải chuẩn — dùng để biết nên nghi
    # ngờ chỗ nào, đừng coi là kết luận. Sai số tính bằng pixel của ảnh gốc,
    # nên cùng một con số ở ảnh 3200px là tương đối nhỏ hơn ở ảnh 1600px.
    if sai_so.mean() > 1.5:
        print("\n!! Sai số tái chiếu cao. Với cảnh gần phẳng (trang sách, tờ "
              "giấy) đây là\n"
              "!! dấu hiệu SfM bị nhập nhằng: mọi điểm nằm trên một mặt phẳng "
              "thì có vô số\n"
              "!! cách đặt máy ảnh cho ra cùng bức ảnh. Nếu chữ mờ, hãy nghi "
              "chỗ này TRƯỚC\n"
              "!! khi nghi thuật toán densification — đổi sang AbsGS cũng "
              "không cứu được\n"
              "!! một bộ pose sai. Cách chữa nằm ở lúc chụp: mở sách cho trang "
              "cong, đổi\n"
              "!! khoảng cách đứng chụp, lót vân ngẫu nhiên bên dưới.")
    elif sai_so.mean() < 0.8:
        print("\nSai số tái chiếu thấp — pose không phải chỗ đáng nghi.")
    if track.mean() < 4:
        print(f"\n!! Mỗi điểm chỉ được {track.mean():.1f} camera nhìn thấy. "
              f"Ràng buộc yếu; ảnh cạnh nhau\n"
              f"!! cần trùng nhau khoảng 60-80%.")
else:
    print("Không có points3D.bin (chỉ có .txt?) — bỏ qua phần đo sai số tái chiếu.")

print()

# Colab free có khoảng 12,7 GB RAM. Chừa ~2 GB cho Python, torch và dữ liệu
# COLMAP thì còn 10,5 GB cho ảnh. T4 có 15 GB VRAM, không phải 16.
TRAN_RAM_GB = 10.5
TRAN_VRAM_GB = 15.0

W, H = Image.open(f"{WORK}/images/{photos[0]}").size
N = len(photos)


def ram_anh_gb(byte_moi_kenh):
    """Chỗ ảnh chiếm khi 3DGS nạp hết vào bộ nhớ, tính theo GB thập phân."""
    return N * W * H * 3 * byte_moi_kenh / 1e9


ram_f32 = ram_anh_gb(4)
ram_u8 = ram_anh_gb(1)

# Ta luôn truyền -r 1 nên 3DGS không hạ ảnh xuống nữa: kích thước dùng để tính
# đúng bằng kích thước ảnh trên đĩa. Với -r -1 của bản cũ thì mọi ảnh rộng hơn
# 1600px bị hạ về 1600px và con số dưới đây sẽ không còn đúng.
if "uint8" in CFG["va"]:
    ram_thuc, nhan = ram_u8, "uint8 (đã vá)"
elif ram_f32 > TRAN_RAM_GB >= ram_u8:
    CFG["va"].insert(0, "uint8")
    ram_thuc, nhan = ram_u8, "uint8 (tự bật thêm — float32 không vừa RAM)"
else:
    ram_thuc, nhan = ram_f32, "float32"

print(f"Ảnh          : {N} tấm, {W}×{H}")
print(f"RAM cho ảnh  : {ram_f32:.2f} GB nếu float32 | {ram_u8:.2f} GB nếu uint8")
print(f"               sẽ dùng {ram_thuc:.2f} GB — {nhan}")
print(f"Trần RAM     : {TRAN_RAM_GB} GB")

if ram_thuc > TRAN_RAM_GB:
    vua = int(TRAN_RAM_GB * 1e9 // (W * H * 3 * (1 if "uint8" in CFG["va"] else 4)))
    raise SystemExit(
        f"\nDỪNG. Ảnh cần {ram_thuc:.2f} GB RAM, trần là {TRAN_RAM_GB} GB.\n"
        f"Ở cỡ {W}×{H} thì tối đa {vua} ảnh, mà đang có {N}.\n"
        f"Chọn một trong hai: bớt ảnh xuống {vua} tấm, hoặc quay lại chặng 0 "
        f"thu nhỏ ảnh rồi chạy lại chặng 1 và 2.\n"
        f"(Nâng độ phân giải chỉ ở chặng 3 là vô nghĩa — nó bị khoá từ chặng 0.)")

# ---- VRAM: mỗi hạt Gaussian giữ tham số + gradient + hai trạng thái Adam ----
sh = int(CFG["cli"].get("--sh_degree", 3))
so_he_so_sh = 3 * (sh + 1) ** 2              # f_dc (3) + f_rest
so_float = 3 + 3 + 4 + 1 + so_he_so_sh       # xyz, scale, rot, opacity, SH
byte_moi_hat = so_float * 4 * 4              # 4 byte/float × (tham số+grad+m+v)

tran_hat = CFG["patch"]["MAX_GAUSSIANS"]
vram_thuong = tran_hat * byte_moi_hat / 1e9
# densify_and_split dựng tensor tạm cỡ bằng tensor thật, nên đỉnh xấp xỉ gấp
# đôi. Cộng ~1,5 GB cho context CUDA và buffer rasterize.
vram_dinh = vram_thuong * 2 + 1.5

print(f"\nMỗi hạt      : {so_float} float (SH bậc {sh}) = {byte_moi_hat} B "
      f"kể cả gradient và Adam")
print(f"Trần số hạt  : {tran_hat:,}")
print(f"VRAM         : {vram_thuong:.1f} GB thường trực, đỉnh khoảng "
      f"{vram_dinh:.1f} GB / {TRAN_VRAM_GB} GB")

if vram_dinh > TRAN_VRAM_GB:
    moi = int((TRAN_VRAM_GB - 1.5) / 2 * 1e9 / byte_moi_hat)
    CFG["patch"]["MAX_GAUSSIANS"] = moi
    print(f"!! Đỉnh vượt trần — hạ trần số hạt xuống {moi:,}")

print("\nCon số VRAM là ước lượng. Bản vá log in max_memory_allocated() thật "
      "mỗi 1000 iter;\nnếu nó vượt 12 GB thì hạ MAX_GAUSSIANS thêm 25%.")

## 6. Install Gaussian Splatting

Takes about 10 minutes, most of it compiling the CUDA rasteriser. Output is kept
quiet unless something fails, in which case the last 30 lines are printed.

**Bộ vẽ có hai nhánh, và mốc bạn chọn ở ô 6 quyết định build nhánh nào.**

| nhánh | `--antialiasing` | `--optimizer_type sparse_adam` |
|---|---|---|
| `dr_aa` — nhánh `.gitmodules` ghim sẵn | có | **không** |
| `3dgs_accel` — nhánh README của Inria bảo checkout | có | có |

`3dgs_accel` là tập cha, không phải một sự đánh đổi. Ô này tự `git checkout`
sang nhánh cần, gỡ bản đã cài và xoá `build/` trước khi biên dịch lại — không
dọn thì setuptools dùng lại đống `.o` của nhánh trước, build "thành công" mà
chạy nhầm bộ vẽ, kiểu hỏng tệ nhất vì không có dòng lỗi nào.

**Build `3dgs_accel` hỏng thì ô này không sập.** Nó lùi về `dr_aa`, gỡ
`--optimizer_type` khỏi cấu hình và giữ `--antialiasing`. Phải gỡ, vì `train.py`
`sys.exit` ngay dòng đầu khi thấy `sparse_adam` mà không import được
`SparseGaussianAdam`.

`fused-ssim` cũng vậy: hỏng thì `train.py` tự lùi về hàm `ssim` viết bằng
PyTorch, kết quả vẫn đúng, chỉ chậm hơn. Nên nó không được phép làm sập ô.

Inria công bố **×1,6** với `--optimizer_type default` và **×2,7** với
`sparse_adam`. Đó là số họ đo trên máy họ, trên bộ dữ liệu của họ — chưa ai đo
trên T4 free tier ở 3200px với `--data_device cpu`. Mốc `[LOG]` ở ô 8 mới là số
thật của bạn.

In [ ]:
import os
import subprocess

os.environ["TORCH_CUDA_ARCH_LIST"] = ARCH
REPO = "/content/gaussian-splatting"
DGR = f"{REPO}/submodules/diff-gaussian-rasterization"


def chay(command, cwd=None):
    """Chạy một lệnh shell. Trả về (chạy được không, stdout+stderr)."""
    kq = subprocess.run(command, shell=True, capture_output=True, text=True,
                        cwd=cwd)
    return kq.returncode == 0, kq.stdout + kq.stderr


def step(label, command):
    """Run a shell command quietly. On failure, show enough to diagnose it."""
    ok, ra = chay(command)
    if not ok:
        print(f"FAILED: {label}\n")
        print("\n".join(ra.splitlines()[-30:]))
        raise RuntimeError(label)
    print("ok  ", label)


if not os.path.isdir(REPO):
    step("clone repository",
         "git clone -q --recursive "
         f"https://github.com/graphdeco-inria/gaussian-splatting {REPO}")

step("plyfile", "pip -q install plyfile")

# ===========================================================================
# BỘ VẼ — nhánh nào, và vì sao lại có chuyện chọn nhánh
#
# .gitmodules của Inria ghim diff-gaussian-rasterization vào nhánh dr_aa. Nhánh
# đó CÓ antialiasing (trường antialiasing nằm trong GaussianRasterizationSettings)
# nhưng KHÔNG có SparseGaussianAdam. README của Inria bảo muốn sparse_adam thì
# phải checkout sang nhánh 3dgs_accel — và nhánh đó có CẢ hai, nên đây là tập
# cha chứ không phải một sự đánh đổi.
#
# train.py xử rất phũ nếu thiếu:
#     if not SPARSE_ADAM_AVAILABLE and opt.optimizer_type == "sparse_adam":
#         sys.exit(...)
# Nên build hỏng mà cứ để cờ lại là ô 8 chết ngay dòng đầu. Ô này lùi hẳn về
# dr_aa và GỠ --optimizer_type ra khỏi cấu hình, còn --antialiasing thì giữ,
# vì dr_aa vẫn làm được. Mất một cờ còn hơn sập cả ô.
# ===========================================================================


def dat_nhanh(nhanh):
    """Đưa submodule bộ vẽ về đúng nhánh cần. Trả về (ok, log)."""
    if nhanh == "dr_aa":
        # Về đúng commit mà repo cha GHIM, không phải đầu nhánh. Hôm nay hai
        # thứ đó bằng nhau, nhưng không có gì bảo đảm ngày mai còn bằng, và
        # commit được ghim mới là thứ đã đi cùng train.py này.
        return chay("git submodule update --force --init "
                    "submodules/diff-gaussian-rasterization", cwd=REPO)
    ok, ra = chay(f"git fetch -q origin {nhanh}", cwd=DGR)
    if not ok:
        return False, ra
    return chay("git checkout -q -f FETCH_HEAD", cwd=DGR)


def cai_bo_ve(nhanh):
    """Checkout nhánh rồi build lại bộ vẽ từ đầu. Trả về (ok, log)."""
    ok, ra = dat_nhanh(nhanh)
    if not ok:
        return False, ra
    # Dọn bản đã cài và thư mục build cũ. Không dọn thì setuptools dùng lại
    # đống .o của nhánh trước, build "thành công" mà chạy nhầm bộ vẽ — kiểu
    # hỏng tệ nhất, vì không có dòng lỗi nào cả.
    chay("pip -q uninstall diff-gaussian-rasterization -y")
    chay(f"rm -rf {DGR}/build")
    return chay(f"pip -q install {DGR}")


NHANH = CFG["rasterizer"]
print(f"Bộ vẽ cần   : {NHANH}  (mốc {MILESTONE})")
print("Đang build, khoảng 10 phút...")

ok, log = cai_bo_ve(NHANH)

if not ok and NHANH == "3dgs_accel":
    print("\n!! Build 3dgs_accel HỎNG. Ba mươi dòng cuối:\n")
    print("\n".join(log.splitlines()[-30:]))
    print("\n!! Lùi về dr_aa: giữ --antialiasing, bỏ --optimizer_type.")
    NHANH = "dr_aa"
    CFG["rasterizer"] = "dr_aa"
    CFG["cli"].pop("--optimizer_type", None)
    ok, log = cai_bo_ve(NHANH)

if not ok:
    print("FAILED: diff-gaussian-rasterization\n")
    print("\n".join(log.splitlines()[-30:]))
    raise RuntimeError("diff-gaussian-rasterization")
print("ok   diff-gaussian-rasterization", f"({NHANH})")

step("simple-knn", f"pip -q install {REPO}/submodules/simple-knn")

# fused-ssim là nửa còn lại của phần tăng tốc. Thiếu nó thì train.py tự lùi về
# hàm ssim viết bằng PyTorch (try/except quanh chỗ import), chạy vẫn ra kết quả
# đúng, chỉ là chậm hơn. Nên nó không được phép làm sập ô này.
if os.path.isdir(f"{REPO}/submodules/fused-ssim"):
    ok, log = chay(f"pip -q install {REPO}/submodules/fused-ssim")
    if ok:
        print("ok   fused-ssim")
    else:
        print("!!   fused-ssim build hỏng — train.py sẽ dùng ssim của PyTorch,")
        print("!!   kết quả vẫn đúng, chỉ chậm hơn. Ba dòng cuối:")
        print("\n".join(log.splitlines()[-3:]))

# ---- Xác nhận bằng import thật, không tin vào việc build đã chạy xong -------
import torch
from diff_gaussian_rasterization import (GaussianRasterizationSettings,
                                         GaussianRasterizer)  # noqa: F401

# GaussianRasterizationSettings là NamedTuple, và gaussian_renderer luôn truyền
# antialiasing=pipe.antialiasing vào đó. Bộ vẽ thiếu trường này thì chết ngay
# lời gọi render đầu tiên, không phải nửa tiếng sau.
CO_AA = "antialiasing" in GaussianRasterizationSettings._fields

try:
    from diff_gaussian_rasterization import SparseGaussianAdam  # noqa: F401
    CO_SPARSE_ADAM = True
except Exception:
    CO_SPARSE_ADAM = False

try:
    import fused_ssim  # noqa: F401
    CO_FUSED_SSIM = True
except Exception:
    CO_FUSED_SSIM = False

# ---- Cấu hình phải khớp với thứ vừa cài được, không phải thứ ta mong muốn ---
if "--antialiasing" in CFG["co_bat"] and not CO_AA:
    CFG["co_bat"].remove("--antialiasing")
    CFG["co_render"] = [c for c in CFG["co_render"] if c != "--antialiasing"]
    print("!!   Bộ vẽ này không có trường antialiasing — đã bỏ cờ khỏi lệnh train.")

if CFG["cli"].get("--optimizer_type") == "sparse_adam" and not CO_SPARSE_ADAM:
    CFG["cli"].pop("--optimizer_type")
    print("!!   Không import được SparseGaussianAdam — đã bỏ --optimizer_type.")
    print("!!   Để lại là train.py sys.exit ngay dòng đầu của ô 8.")

print()
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("bộ vẽ        :", NHANH,
      "| antialiasing", CO_AA, "| SparseGaussianAdam", CO_SPARSE_ADAM)
print("fused-ssim   :", CO_FUSED_SSIM,
      "(thiếu thì chậm hơn, kết quả không đổi)")
# Đây là lý do mốc V2a phải tồn tại: train.py truyền
# separate_sh=SPARSE_ADAM_AVAILABLE, tức là đường tính SH đổi theo việc CÀI
# ĐƯỢC bộ vẽ, không phải theo cờ --optimizer_type. Cài xong là đã khác nền rồi.
print("separate_sh  :", CO_SPARSE_ADAM,
      "— bật theo việc cài được bộ vẽ, KHÔNG theo cờ --optimizer_type")
print("\nSẽ train với:", " ".join(
    [f"{k} {v}" for k, v in CFG["cli"].items()] + CFG["co_bat"]))
print("Ready to train.")

## 7. Vá mã nguồn 3DGS

Bốn thứ dưới đây không có đường CLI nào tới được, phải sửa thẳng vào mã nguồn.
Chạy sau ô cài đặt, trước ô train.

| | vá gì | vì sao |
|---|---|---|
| 1 | ảnh gốc giữ ở dạng `uint8` | 1 byte mỗi kênh thay vì 4; đi cùng `--data_device cpu` thì ảnh nằm hẳn ở RAM |
| 2 | trần cứng số hạt Gaussian | bản Inria **không có** `max_cap` — đó là thứ của gsplat/MCMC. Hằng số `min_opacity` cũng viết cứng trong `train.py` |
| 3 | phạt hạt kéo dài trong mặt phẳng | chỉ phạt `s1/s2`, **không** phạt `s_max/s_min`: hạt dẹt như cái đĩa áp sát mặt phẳng là thứ đang đúng, phạt nó là đánh nhầm |
| 4 | log số hạt + VRAM đỉnh mỗi 1000 iter | không có nó thì không debug nổi OOM trên Colab |

Ô này chạy lại được: nó trả `train.py` về bản `.bak` rồi vá lại từ đầu, nên đổi
`PRESET` xong chạy lại là ra đúng bộ vá mới. Vá xong có kiểm tra cú pháp; hỏng
thì tự trả file về bản gốc chứ không để train một bản vỡ.

Cuối ô có bước đối chiếu mọi cờ sắp truyền với `python train.py --help` thật —
cả cờ có giá trị lẫn cờ bật/tắt. Gõ nhầm tên cờ thì biết ngay ở đây, không phải
đợi tới ô 8.

In [ ]:
import ast
import os
import re
import shutil
import subprocess

REPO = "/content/gaussian-splatting"
TRAIN = f"{REPO}/train.py"
RENDER = f"{REPO}/render.py"
CAM = f"{REPO}/scene/cameras.py"


def doc(duong):
    with open(duong, encoding="utf-8") as f:
        return f.read()


def ghi(duong, chu):
    with open(duong, "w", encoding="utf-8") as f:
        f.write(chu)


def thut_le(chu, vi_tri):
    """Mức thụt lề của dòng chứa vị trí này. None nếu có chữ khác đứng trước."""
    dau = chu.rfind("\n", 0, vi_tri) + 1
    dem = chu[dau:vi_tri]
    return dem if not dem.strip() else None


# Chạy lại ô này bao nhiêu lần cũng được: có .bak thì trả file về bản gốc rồi
# vá lại từ đầu, nên đổi PRESET xong chạy lại là ra đúng bộ vá mới chứ không
# chồng lên bộ cũ. Dùng Python chứ không dùng sed — sed vỡ khi thụt lề đổi.
for duong in (TRAIN, RENDER, CAM):
    if os.path.exists(duong + ".bak"):
        shutil.copy(duong + ".bak", duong)
    else:
        shutil.copy(duong, duong + ".bak")

src = doc(TRAIN)
bao_cao = []


def ket(ten, chu):
    bao_cao.append((ten, chu))
    print(f"  {ten:<6} {chu}")


print("Đang vá", TRAIN)

# ---- VÁ 1: giữ ảnh gốc ở dạng uint8 ---------------------------------------
# Cùng với --data_device cpu, ảnh nằm ở RAM thay vì VRAM và nhẹ đi bốn lần.
UINT8 = '''"""
Giữ ảnh gốc trong Camera ở dạng uint8 thay vì float32.

Một byte mỗi kênh thay vì bốn, và chỉ đổi về float đúng lúc dùng. Đi kèm
--data_device cpu thì ảnh nằm hẳn ở RAM, trả lại gần như trọn VRAM cho hạt
Gaussian, optimizer và rasterizer.
"""
import torch
from scene.cameras import Camera


def _lay(self):
    if getattr(self, "_anh_u8", None) is None:
        return None
    return self._anh_u8.to("cuda", non_blocking=True).float().div_(255.0)


def _dat(self, gia_tri):
    if gia_tri is None:
        self._anh_u8 = None
        return
    self._anh_u8 = (gia_tri.detach().clamp(0.0, 1.0)
                    .mul(255.0).round().to(torch.uint8).cpu())


# property có setter là data descriptor, nên nó thắng cả self.original_image = x
# ở trong Camera.__init__ — không phải sửa cameras.py.
Camera.original_image = property(_lay, _dat)
'''

if "uint8" in CFG["va"]:
    if "original_image" not in doc(CAM):
        ket("VÁ 1", "BỎ QUA — scene/cameras.py không còn thuộc tính "
                    "original_image. Bản 3DGS này đã đổi cấu trúc Camera; "
                    "đọc lại cameras.py trước khi vá mù.")
    else:
        ghi(f"{REPO}/uint8_patch.py", UINT8)
        ast.parse(UINT8)
        src = "import uint8_patch  # V2 VÁ 1\n" + src
        # render.py cũng nạp cả bộ ảnh vào bộ nhớ. Ở 3200px mà không vá thì ô
        # đo PSNR/SSIM chết vì hết RAM, dù train đã chạy xong êm xuôi.
        ghi(RENDER, "import uint8_patch  # V2 VÁ 1\n" + doc(RENDER))
        ket("VÁ 1", "ảnh gốc chuyển sang uint8 (uint8_patch.py, import vào "
                    "train.py và render.py)")
else:
    ket("VÁ 1", "không bật")

# ---- VÁ 2: trần cứng số hạt Gaussian --------------------------------------
# 3DGS gốc không có max_cap — đó là thứ của gsplat/MCMC, không phải của bản
# Inria này. Muốn có trần thì phải vá. Con số min_opacity trong lời gọi cũng
# là hằng số viết cứng, không có đường CLI nào tới được.
if "cap" in CFG["va"]:
    vi_tri = src.find("gaussians.densify_and_prune(")
    thut = thut_le(src, vi_tri) if vi_tri >= 0 else None
    if thut is None:
        ket("VÁ 2", "BỎ QUA — không tìm thấy lời gọi densify_and_prune đứng "
                    "riêng một dòng trong train.py")
    else:
        # Đếm ngoặc để lấy trọn lời gọi, kể cả khi nó trải nhiều dòng.
        mo = src.index("(", vi_tri)
        sau = 0
        for cuoi in range(mo, len(src)):
            if src[cuoi] == "(":
                sau += 1
            elif src[cuoi] == ")":
                sau -= 1
                if sau == 0:
                    break
        goi = src[vi_tri:cuoi + 1]
        MAX_G = CFG["patch"]["MAX_GAUSSIANS"]
        MIN_OP = CFG["patch"]["MIN_OPACITY"]
        # Tham số thứ hai là min_opacity, viết cứng 0.005 trong bản gốc.
        # Đếm số lần thay chứ không so chuỗi trước/sau: preset OBJECT đặt
        # MIN_OPACITY đúng bằng 0.005 của bản gốc, so chuỗi sẽ ra "không đổi"
        # rồi báo động giả.
        goi_moi, so_lan = re.subn(r"(densify_and_prune\(\s*[^,]+,\s*)[0-9.eE+-]+",
                                  lambda m: m.group(1) + repr(MIN_OP), goi, count=1)
        if so_lan == 0:
            ket("VÁ 2", "CẢNH BÁO — không nhận ra tham số min_opacity, giữ "
                        "nguyên hằng số gốc; phần trần số hạt vẫn được vá")
        dau_dong = src.rfind("\n", 0, vi_tri) + 1
        moi = (f"{thut}# V2 VÁ 2 — trần cứng số hạt. Chạm trần thì ngừng sinh, chỉ tỉa.\n"
               f"{thut}if gaussians.get_xyz.shape[0] < {MAX_G}:\n"
               f"{thut}    {goi_moi}\n"
               f"{thut}else:\n"
               f"{thut}    gaussians.prune_points("
               f"(gaussians.get_opacity < {MIN_OP}).squeeze())\n"
               f"{thut}    torch.cuda.empty_cache()")
        src = src[:dau_dong] + moi + src[cuoi + 1:]
        ket("VÁ 2", f"trần {MAX_G:,} hạt, min_opacity {MIN_OP}")
else:
    ket("VÁ 2", "không bật")

# ---- VÁ 3: phạt hạt bị kéo dài TRONG mặt phẳng -----------------------------
# Không phạt s_max/s_min. Với mặt phẳng thì hạt NÊN dẹt như cái đĩa áp sát bề
# mặt — hai trục lớn, một trục siêu mỏng. Phạt s_max/s_min là đánh thẳng vào
# cái đang đúng. Thứ cần triệt là một trục dài hai trục ngắn, tức là s1/s2.
vi_tri = src.find("loss.backward()")
thut = thut_le(src, vi_tri) if vi_tri >= 0 else None

if "aniso" in CFG["va"]:
    if thut is None:
        ket("VÁ 3", "BỎ QUA — không tìm thấy loss.backward() đứng riêng dòng")
    else:
        lam = CFG["patch"]["ANISO_LAMBDA"]
        ti_le = CFG["patch"]["ANISO_MAX_RATIO"]
        dau_dong = src.rfind("\n", 0, vi_tri) + 1
        chen = (f"{thut}# V2 VÁ 3 — chỉ phạt tỉ lệ hai trục LỚN nhất, thả tự do trục mỏng\n"
                f"{thut}_s = torch.sort(gaussians.get_scaling, dim=1, descending=True)[0]\n"
                f"{thut}_ti_le = _s[:, 0] / (_s[:, 1] + 1e-8)\n"
                f"{thut}loss = loss + {lam} * torch.clamp("
                f"_ti_le - {ti_le}, min=0.0).mean()\n")
        src = src[:dau_dong] + chen + src[dau_dong:]
        ket("VÁ 3", f"phạt s1/s2 vượt {ti_le}, lambda {lam}")
else:
    ket("VÁ 3", "không bật (chỉ FLAT_OBJECT dùng)")

# ---- VÁ 4: log số hạt và VRAM đỉnh ----------------------------------------
# Ô train chỉ đọc dòng cuối của train.log mỗi 30 giây, nên thông tin phải nằm
# trong log mới thấy được. Không có nó thì không debug nổi OOM trên Colab.
vi_tri = src.find("loss.backward()")
thut = thut_le(src, vi_tri) if vi_tri >= 0 else None

if "log" in CFG["va"]:
    if thut is None:
        ket("VÁ 4", "BỎ QUA — không tìm thấy loss.backward() đứng riêng dòng")
    else:
        cuoi_dong = src.find("\n", vi_tri) + 1
        chen = (f'{thut}# V2 VÁ 4 — mốc theo dõi, ô train đọc lại từ train.log\n'
                f'{thut}if iteration % 1000 == 0 or iteration == 1:\n'
                f'{thut}    print(f"[LOG] iter {{iteration}}"\n'
                f'{thut}          f" | hat {{gaussians.get_xyz.shape[0]}}"\n'
                f'{thut}          f" | vram_dinh {{torch.cuda.max_memory_allocated() / 1e9:.2f}} GB"\n'
                f'{thut}          f" | loss {{loss.item():.5f}}", flush=True)\n')
        src = src[:cuoi_dong] + chen + src[cuoi_dong:]
        ket("VÁ 4", "in số hạt + VRAM đỉnh + loss mỗi 1000 iter")
else:
    ket("VÁ 4", "không bật")

# ---- Kiểm tra hậu vá -------------------------------------------------------
try:
    ast.parse(src)
except SyntaxError as e:
    ghi(TRAIN, doc(TRAIN + ".bak"))
    raise SystemExit(f"Vá hỏng cú pháp train.py dòng {e.lineno}: {e.msg}. "
                     f"Đã trả file về bản gốc, chưa train gì cả.")
ghi(TRAIN, src)

print("\ntrain.py vá xong, cú pháp hợp lệ. Bản gốc nằm ở train.py.bak")

# Mọi tham số CLI phải thật sự tồn tại trong bản 3DGS vừa clone. Đọc --help
# chứ không đoán: các bản 3DGS khác nhau có bộ tham số khác nhau.
kq = subprocess.run(["python", "train.py", "--help"], cwd=REPO,
                    capture_output=True, text=True)
if kq.returncode != 0 or not kq.stdout.strip():
    print("!! Không chạy được `train.py --help` nên chưa đối chiếu được tham "
          "số CLI:")
    print("\n".join((kq.stdout + kq.stderr).splitlines()[-10:]))
else:
    # Cả cờ có giá trị lẫn cờ bật/tắt. Ba cờ của bản tháng 10/2024
    # (--antialiasing, --optimizer_type) đi qua đúng cửa kiểm tra này, nên
    # không có chuyện gõ nhầm tên cờ mà tới ô 8 mới biết.
    can_kiem = list(CFG["cli"]) + CFG.get("co_bat", [])
    thieu = [co for co in can_kiem if co not in kq.stdout]
    if thieu:
        raise SystemExit(f"train.py của bản này không có tham số {thieu}. "
                         f"Chạy `python train.py --help` trong repo mà đối chiếu.")
    print("Đã đối chiếu", len(can_kiem),
          "tham số CLI với train.py --help — đủ cả.")

print("\nĐoạn đã vá:")
for dong in src.splitlines():
    if "V2 VÁ" in dong:
        print("   ", dong.strip())

## 8. Train

Each checkpoint is copied to Drive as soon as it lands on disk. If Colab drops
the session halfway, everything already copied is still yours.

Dòng in ra mỗi 30 giây là mốc `[LOG]` của bản vá 4: số hạt hiện có và VRAM đỉnh.
Hai con số đó là thứ duy nhất nói trước được sắp hết bộ nhớ hay chưa.

In [ ]:
import glob
import os
import shutil
import subprocess
import time

os.makedirs(OUTPUT_DIR, exist_ok=True)

command = ["python", "train.py",
           "-s", WORK,
           "-m", f"{WORK}/output",
           "--save_iterations"] + [str(s) for s in SAVES]

for co, gia_tri in CFG["cli"].items():
    command += [co, str(gia_tri)]

# Cờ bật/tắt đi riêng: argparse store_true không nhận giá trị, đưa thêm chữ
# "True" vào là nó coi đó là tham số thừa rồi chết.
command += CFG.get("co_bat", [])

if EVAL:
    # 3DGS giữ lại 1/8 số ảnh, không đưa vào train, để ô 9 đo PSNR/SSIM.
    command.append("--eval")

print("$", " ".join(command), "\n")

log_path = f"{WORK}/train.log"
log_file = open(log_path, "w")
trainer = subprocess.Popen(command, cwd="/content/gaussian-splatting",
                           stdout=log_file, stderr=subprocess.STDOUT, text=True)
print("Training started. A progress line follows every 30 seconds.\n")

copied = set()


def copy_new_checkpoints():
    """Copy any checkpoint that has appeared since the last look."""
    pattern = f"{WORK}/output/point_cloud/iteration_*/point_cloud.ply"
    for path in glob.glob(pattern):
        step_number = path.split("iteration_")[1].split("/")[0]
        if step_number in copied:
            continue
        time.sleep(5)          # let the file finish being written
        target = f"{OUTPUT_DIR}/{PROJECT}_{step_number}.ply"
        shutil.copy(path, target)
        copied.add(step_number)
        size = os.path.getsize(target) / 1e6
        print(f">>> SAVED TO DRIVE: {PROJECT}_{step_number}.ply ({size:.0f} MB)")


def dong_moi_nhat():
    """Dòng đáng đọc nhất trong log.

    Ưu tiên mốc [LOG] của bản vá 4 — nó có số hạt và VRAM đỉnh, tức là hai
    con số duy nhất nói trước được sắp OOM hay chưa. Thanh tiến độ của tqdm
    đi ra bằng ký tự xuống dòng \\r nên phải tách ra mới đọc được.
    """
    try:
        with open(log_path, encoding="utf-8", errors="replace") as f:
            chu = f.read()
    except OSError:
        return ""
    dong = [d.strip() for d in chu.replace("\r", "\n").splitlines() if d.strip()]
    moc = [d for d in dong if "[LOG]" in d]
    return moc[-1] if moc else (dong[-1] if dong else "")


while trainer.poll() is None:
    time.sleep(30)
    copy_new_checkpoints()
    tail = dong_moi_nhat()
    if tail:
        print(tail)

log_file.close()
copy_new_checkpoints()

if not copied:
    print("\nNo checkpoint was produced. Last 30 lines of the log:\n")
    print(subprocess.run(["tail", "-30", log_path],
                         capture_output=True, text=True).stdout)
else:
    print("\nFinished. Saved:", sorted(copied, key=int))
    with open(log_path, encoding="utf-8", errors="replace") as f:
        moc = [d.strip() for d in f.read().replace("\r", "\n").splitlines()
               if "[LOG]" in d]
    if moc:
        print("Mốc đầu :", moc[0])
        print("Mốc cuối:", moc[-1])

        # ===================================================================
        # ĐO C — trần số hạt có phải là thứ đang chặn không?
        #
        # Nếu số hạt dừng đúng ở trần thì thứ giới hạn chi tiết là BỘ NHỚ,
        # không phải thuật toán densification. Trong ca đó, đổi sang AbsGS
        # hay Improved-GS là chữa nhầm bệnh — nâng trần mới là việc phải làm.
        # ===================================================================
        import re as _re

        so_hat = [int(m.group(1))
                  for d in moc if (m := _re.search(r"hat (\d+)", d))]
        dinh = [float(m.group(1))
                for d in moc if (m := _re.search(r"vram_dinh ([\d.]+)", d))]
        tran = CFG["patch"]["MAX_GAUSSIANS"]

        if so_hat:
            cuoi = so_hat[-1]
            print(f"\nSố hạt cuối : {cuoi:,} / trần {tran:,} "
                  f"({cuoi / tran * 100:.0f}%)")
            if cuoi >= tran * 0.98:
                print("!! CHẠM TRẦN. Thứ đang chặn chi tiết là bộ nhớ, không "
                      "phải thuật toán.")
                print("!! Nâng MAX_GAUSSIANS trong bảng preset ở ô 6 rồi chạy "
                      "lại — đừng đổi")
                print("!! thuật toán densification, nó không phải chỗ hỏng.")
                if dinh:
                    con = 15.0 - dinh[-1]
                    print(f"!! VRAM đỉnh đang là {dinh[-1]:.1f} GB, còn trống "
                          f"khoảng {con:.1f} GB trên T4.")
            elif cuoi < tran * 0.6:
                print("=> Còn xa trần. Bộ nhớ không phải chỗ chặn — densification "
                      "tự dừng lại")
                print("   vì không hạt nào đạt ngưỡng nữa. Đây đúng là ca "
                      "gradient collision mà")
                print("   AbsGS sinh ra để chữa, NẾU ĐO B cho thấy ảnh train "
                      "vẫn nét.")
        if dinh:
            print(f"VRAM đỉnh   : {max(dinh):.2f} GB / 15 GB")
            if max(dinh) > 12.0:
                print("!! Trên 12 GB — hạ MAX_GAUSSIANS thêm 25% rồi chạy lại.")
    # Log đi kèm kết quả: hai tuần nữa không ai nhớ lần chạy này đặt tham số gì.
    shutil.copy(log_path, f"{OUTPUT_DIR}/{PROJECT}_train.log")

## 9. Đo — trần thật sự nằm ở đâu

Không có phép đo thì mọi kết luận chỉ là cảm giác. Ô này trả lời hai câu khác
nhau, và tách chúng ra là điểm mấu chốt.

**ĐO B — mô hình có tái tạo nổi thứ nó được nhìn thẳng vào không?** Render lại
một ảnh **train** rồi so với chính tấm ảnh gốc, đo phần năng lượng tần số cao
còn giữ lại được ở vùng bạn chỉ định.

| kết quả | trần nằm ở đâu | làm gì tiếp |
|---|---|---|
| chữ nét trên ảnh train, mờ trên ảnh đối chứng | densification | AbsGS / Improved-GS nhắm đúng bệnh này |
| chữ mờ ngay trên ảnh train | pose, hoặc trần số hạt, hoặc độ phân giải | đổi thuật toán **không** cứu được — xem ĐO A ở ô 5 và ĐO C ở cuối ô 8 |

Nhớ đặt `SOI_TAM` vào đúng vùng có chữ. Vùng giấy trắng không có chữ lúc nào
cũng cho tỉ lệ cao và không nói lên điều gì.

**PSNR/SSIM trên bộ đối chứng** — chạy khi `EVAL = True`. Các mốc phải đo tách
bạch, mỗi lần **một** thay đổi, không gộp:

| mốc | so với mốc trên, đổi thêm gì |
|---|---|
| V1 | bản cũ: `LIMIT_VRAM=True`, ảnh nằm trên VRAM |
| **V1_5** | bỏ `LIMIT_VRAM` + `--data_device cpu`. Vẫn 1600px, bộ vẽ `dr_aa` |
| **V2a** | 3200px ở chặng 0 + bảng preset + các bản vá + bộ vẽ `3dgs_accel` |
| **V2b** | `--antialiasing` |
| **V2c** | `--optimizer_type sparse_adam` |

`V2a` là nền, không phải mốc thừa. `train.py` truyền
`separate_sh=SPARSE_ADAM_AVAILABLE`, nên chỉ **cài được** `3dgs_accel` là đường
tính SH đã đổi, chưa bật cờ nào. Nhảy thẳng từ `V1_5` sang `V2b` là gộp bộ vẽ
với cờ vào một con số rồi không tách ra được nữa.

Đổi `MILESTONE` ở ô 6 rồi chạy lại từ ô cài đặt (phải build lại bộ vẽ). Tên dự án tự
đeo hậu tố mốc nên các mốc không đè file của nhau. Rất có thể V1_5 đã giải quyết
xong chuyện chữ mờ — biết được điều đó tiết kiệm cả buổi.

> **Ô này truyền lại `--antialiasing` cho `render.py`, và đó không phải chuyện
> thừa.** `render.py` dựng `PipelineParams(parser)` không có `sentinel`, nên
> `antialiasing` mặc định là `False` chứ không phải `None`; `get_combined_args`
> chỉ bỏ qua giá trị `None`, nên dòng lệnh **đè** giá trị `True` đã ghi trong
> `cfg_args` lúc train. Không truyền lại thì mọi con số dưới đây là số của một
> mô hình khác với mô hình bạn vừa train.

In [ ]:
import glob
import json
import os
import shutil
import subprocess

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# ---- Chọn tấm nào để soi, cắt vùng nào ------------------------------------
SOI_INDEX = None    # số thứ tự ảnh train muốn soi. None = tấm chính giữa bộ
SOI_CROP = 512      # cạnh ô vuông cắt ra, xem ở tỉ lệ 1:1
SOI_TAM = None      # (x, y) tâm vùng cắt trong ảnh. None = giữa ảnh
# Đặt SOI_TAM vào đúng vùng có chữ thì phép đo dưới đây mới nói được điều
# đáng nói. Giữa ảnh chỉ là điểm khởi đầu.


# ===========================================================================
# render.py NUỐT MẤT --antialiasing — phải truyền lại bằng tay
#
# render.py dựng PipelineParams(parser) mà KHÔNG có sentinel, nên antialiasing
# mặc định là False chứ không phải None. Rồi get_combined_args làm thế này:
#
#     merged_dict = vars(args_cfgfile).copy()      # đọc từ cfg_args của model
#     for k, v in vars(args_cmdline).items():
#         if v != None:
#             merged_dict[k] = v
#
# False khác None, nên dòng lệnh ĐÈ giá trị True đã ghi trong cfg_args lúc
# train. Không truyền lại thì ô này render bằng bộ lọc khác hẳn lúc train, và
# mọi con số PSNR/SSIM/độ nét bên dưới đều là số của một mô hình khác.
#
# --train_test_exp không dính vào chuyện này vì ModelParams dựng với
# sentinel=True, giá trị thành None nên cfg_args thắng. Đợt này ta cũng không
# dùng cờ đó.
#
# --optimizer_type thì KHÔNG được truyền cho render.py: nó thuộc
# OptimizationParams, mà render.py chỉ dựng ModelParams + PipelineParams.
# Việc lọc đã làm ở ô 6 (CO_RENDER_HOP_LE).
# ===========================================================================
CO_RENDER = CFG.get("co_render", [])
print("Cờ truyền lại cho render.py:",
      " ".join(CO_RENDER) if CO_RENDER else "(không có)")
def chay(nhan, lenh):
    kq = subprocess.run(lenh, cwd="/content/gaussian-splatting",
                        capture_output=True, text=True)
    if kq.returncode != 0:
        print(f"FAILED: {nhan}\n")
        print("\n".join((kq.stdout + kq.stderr).splitlines()[-30:]))
        raise RuntimeError(nhan)
    print("ok  ", nhan)


def do_net(anh):
    """
    Năng lượng tần số cao của một mảng ảnh: phương sai của Laplacian.

    Không phải thước đo tuyệt đối — chỉ có ý nghĩa khi đem tỉ lệ giữa ảnh
    render và ảnh gốc của CÙNG một khung hình.

    CÁI BẪY: nhiễu cũng là tần số cao. Một ảnh render lấm tấm nhiễu mà không
    hề nét hơn vẫn cho tỉ lệ cao, có khi vượt 100%. Điều này đáng nhớ vì
    AbsGS nổi tiếng là ảnh hơi bẩn — nếu sau này so AbsGS với bản hiện tại
    bằng con số này mà không nhìn ảnh, rất dễ kết luận ngược. Luôn đọc kèm
    PSNR: nét thật thì tỉ lệ lên VÀ PSNR lên; nhiễu thì tỉ lệ lên còn PSNR
    xuống.
    """
    x = anh.mean(axis=2)
    lap = (-4 * x[1:-1, 1:-1] + x[:-2, 1:-1] + x[2:, 1:-1]
           + x[1:-1, :-2] + x[1:-1, 2:])
    return float(lap.var())


def doc(duong):
    return np.asarray(Image.open(duong).convert("RGB"), dtype=np.float64) / 255.0


def psnr(a, b):
    mse = ((a - b) ** 2).mean()
    return float("inf") if mse == 0 else 10 * np.log10(1.0 / mse)


# ===========================================================================
# ĐO B — mô hình có tái tạo nổi thứ nó được nhìn THẲNG VÀO không?
#
# Đây là phép đo tách bạch được hai loại trần hoàn toàn khác nhau, mà nhìn
# ảnh bằng mắt thì không tách được:
#
#   Chữ nét trên ảnh TRAIN nhưng mờ trên ảnh đối chứng
#       -> mô hình đủ sức biểu diễn, chỉ là không tổng quát hoá được.
#          Trần nằm ở densification. AbsGS / Improved-GS đáng tiền.
#
#   Chữ mờ ngay trên chính ảnh TRAIN
#       -> mô hình không dựng lại nổi tấm ảnh nó được nhìn thẳng vào và được
#          phép khớp thẳng vào. Đây KHÔNG phải chuyện chọn hạt nào để tách.
#          Trần nằm ở pose (xem ĐO A ở ô 5) hoặc ở trần số hạt (ĐO C ở ô 8).
#          Đổi thuật toán densification sẽ không cứu được.
# ===========================================================================
chay("render ảnh train", ["python", "render.py", "-m", f"{WORK}/output",
                          "--skip_test"] + CO_RENDER)

thu_muc = sorted(glob.glob(f"{WORK}/output/train/ours_*"))
if not thu_muc:
    raise SystemExit("Không thấy thư mục train/ours_* — render.py chưa chạy xong?")
goc = next((d for d in thu_muc if d.endswith(f"_{ITERATIONS}")), thu_muc[-1])

renders = sorted(glob.glob(f"{goc}/renders/*.png"))
gts = sorted(glob.glob(f"{goc}/gt/*.png"))
if not renders or len(renders) != len(gts):
    raise SystemExit(f"renders/ và gt/ không khớp nhau trong {goc}")

i = len(renders) // 2 if SOI_INDEX is None else int(SOI_INDEX)
i = max(0, min(i, len(renders) - 1))
anh_ra, anh_goc = doc(renders[i]), doc(gts[i])

H, W = anh_goc.shape[:2]
cx, cy = SOI_TAM if SOI_TAM else (W // 2, H // 2)
nua = SOI_CROP // 2
x0, y0 = max(0, min(cx - nua, W - SOI_CROP)), max(0, min(cy - nua, H - SOI_CROP))
x1, y1 = min(W, x0 + SOI_CROP), min(H, y0 + SOI_CROP)
cat_ra, cat_goc = anh_ra[y0:y1, x0:x1], anh_goc[y0:y1, x0:x1]

net_goc, net_ra = do_net(cat_goc), do_net(cat_ra)
ti_le = net_ra / net_goc if net_goc > 0 else 0.0

print(f"\nẢnh train số {i} / {len(renders) - 1}  ({os.path.basename(gts[i])})")
print(f"Vùng soi     : {SOI_CROP}×{SOI_CROP} px tại ({x0}, {y0})")
print(f"PSNR cả ảnh  : {psnr(anh_ra, anh_goc):.2f} dB")
print(f"PSNR vùng soi: {psnr(cat_ra, cat_goc):.2f} dB")
print(f"Độ nét vùng soi : ảnh gốc {net_goc:.5f} · render {net_ra:.5f} "
      f"· giữ lại {ti_le * 100:.0f}%")

# Ngưỡng dưới đây là kinh nghiệm, không phải chuẩn. Đọc kèm ảnh bên dưới chứ
# đừng đọc một mình.
if ti_le > 0.7:
    print("\n=> Trên ảnh train, mô hình giữ được phần lớn chi tiết. Nếu ảnh đối")
    print("   chứng vẫn mờ thì trần nằm ở TỔNG QUÁT HOÁ, tức densification —")
    print("   đây là ca mà AbsGS / Improved-GS nhắm đúng bệnh.")
elif ti_le < 0.4:
    print("\n=> Mô hình KHÔNG dựng lại nổi tấm ảnh nó được khớp thẳng vào.")
    print("   Đổi thuật toán densification sẽ không cứu được. Xem lại:")
    print("     · ĐO A ở ô 5 — sai số tái chiếu có cao không (pose)")
    print("     · ĐO C ở cuối ô 8 — số hạt có chạm trần không (bộ nhớ)")
    print("     · độ phân giải ảnh: chữ có còn đủ pixel để đọc ở ảnh gốc không")
else:
    print("\n=> Vùng giữa. Đặt SOI_TAM vào đúng chỗ có chữ rồi chạy lại ô này —")
    print("   vùng phẳng không có chữ luôn cho tỉ lệ cao và không nói lên gì.")

print("\nCon số này dùng để SO SÁNH thì chắc hơn là để kết luận một mình.")
print("Giữ nguyên SOI_INDEX và SOI_TAM giữa các mốc, rồi so tỉ lệ của V1_5 với")
print("V2 trên cùng một vùng — đó mới là phép đo nói được mốc nào ăn tiền.")

hinh, truc = plt.subplots(1, 2, figsize=(14, 7))
for ax, anh, ten in ((truc[0], cat_goc, "ảnh gốc"),
                     (truc[1], cat_ra, f"render (giữ {ti_le * 100:.0f}% độ nét)")):
    ax.imshow(np.clip(anh, 0, 1))
    ax.set_title(ten)
    ax.axis("off")
plt.tight_layout()
plt.show()

os.makedirs(OUTPUT_DIR, exist_ok=True)
canh_nhau = np.concatenate([cat_goc, cat_ra], axis=1)
Image.fromarray((np.clip(canh_nhau, 0, 1) * 255).astype(np.uint8)).save(
    f"{OUTPUT_DIR}/{PROJECT}_soi_train.png")
print(f"\nĐã lưu ảnh so sánh: {PROJECT}_soi_train.png")

# ===========================================================================
# ĐO PSNR/SSIM trên bộ đối chứng — cần EVAL = True
# ===========================================================================
if not EVAL:
    print("\nEVAL = False nên không có bộ đối chứng. Bật EVAL ở ô 6 rồi train "
          "lại nếu muốn so V1_5 với V2 bằng số.")
else:
    chay("render ảnh đối chứng", ["python", "render.py", "-m", f"{WORK}/output",
                                  "--skip_train"] + CO_RENDER)
    chay("metrics", ["python", "metrics.py", "-m", f"{WORK}/output"])

    ket_qua = f"{WORK}/output/results.json"
    if os.path.exists(ket_qua):
        with open(ket_qua) as f:
            so = json.load(f)
        print(f"\n{PROJECT}  ({SCAN_TYPE}/{PRESET}, mốc {MILESTONE})")
        for moc, bang in so.items():
            print(f"  {moc:<20} " + "  ".join(f"{k} {v:.4f}"
                                              for k, v in bang.items()))
        shutil.copy(ket_qua, f"{OUTPUT_DIR}/{PROJECT}_metrics.json")
        print(f"\nĐã chép sang Drive: {PROJECT}_metrics.json")

    print("\nPSNR/SSIM nói được cái tổng thể, không nói được chữ có đọc nổi "
          "hay không.\nẢnh render bộ đối chứng nằm ở "
          f"{WORK}/output/test/ours_{ITERATIONS}/renders/ —\nmở đúng một tấm "
          "có vùng chữ, so với ảnh gốc bên cạnh trong gt/, rồi mới kết luận.")

## 10. Make a lighter copy

Throws away the near-transparent blobs and lowers the spherical-harmonics degree
from 3 to `COMPRESS_SH_DEGREE`. The lighter file sits beside the original on
Drive, ending in `_light.ply`.

Useful because the full file often will not open on a modest laptop. Đòn bẩy FPS
lớn nhất trên iGPU là bậc SH chứ không phải số hạt: SH bậc 3 là 48 hệ số mỗi
hạt, hạ về 1 là bỏ 45 trong 48 giá trị.

In [ ]:
import glob
import os

import numpy as np
from plyfile import PlyData, PlyElement


def compress_ply(source, target, sh_degree=1, opacity_threshold=0.05,
                 scale_percentile=None):
    """Drop faint Gaussians and trim the colour detail. Returns (kept, total)."""
    vertices = PlyData.read(source)["vertex"]

    # Stored opacity is pre-sigmoid, so convert before comparing.
    opacity = 1 / (1 + np.exp(-np.asarray(vertices["opacity"])))
    keep = opacity > opacity_threshold

    # Chế độ ROOM bỏ thêm nhóm hạt to bất thường — hạt ma trong phòng gần như
    # luôn là hạt phình rất to và rất mờ. KHÔNG lọc theo khoảng cách tới camera:
    # trong phòng thì tường lúc nào cũng xa, luật đó là thủng tường.
    if scale_percentile is not None and keep.any():
        scale = np.exp(np.stack([np.asarray(vertices[f"scale_{i}"])
                                 for i in range(3)], axis=1))
        lon_nhat = scale.max(axis=1)
        nguong = np.percentile(lon_nhat[keep], scale_percentile)
        keep &= lon_nhat <= nguong

    rest_per_channel = {0: 0, 1: 3, 2: 8, 3: 15}[sh_degree]

    src_names = ["x", "y", "z", "nx", "ny", "nz", "f_dc_0", "f_dc_1", "f_dc_2"]
    dst_names = list(src_names)

    index = 0
    for channel in range(3):
        for i in range(rest_per_channel):
            src_names.append(f"f_rest_{channel * 15 + i}")
            dst_names.append(f"f_rest_{index}")
            index += 1

    tail = ["opacity", "scale_0", "scale_1", "scale_2",
            "rot_0", "rot_1", "rot_2", "rot_3"]
    src_names += tail
    dst_names += tail

    out = np.empty(int(keep.sum()), dtype=[(n, "f4") for n in dst_names])
    for src, dst in zip(src_names, dst_names):
        out[dst] = np.asarray(vertices[src])[keep]

    PlyData([PlyElement.describe(out, "vertex")]).write(target)
    return int(keep.sum()), len(keep)


# Cắt SH sâu hơn giúp FPS trên iGPU nhiều hơn là bớt số hạt: SH bậc 3 là 48 hệ
# số mỗi hạt, hạ về 1 là bỏ 45/48 giá trị — nhẹ cả file lẫn phép tính mỗi khung.
loc = CFG["pipeline"]
print(f"Mức hậu xử lý: {loc['floater']} · bỏ hạt opacity dưới "
      f"{loc['opacity_loc']}" +
      (f", và hạt to hơn phân vị {loc['scale_percentile']}%"
       if loc["scale_percentile"] else "") +
      f" · giữ SH bậc {COMPRESS_SH_DEGREE}\n")

for source in sorted(glob.glob(f"{OUTPUT_DIR}/{PROJECT}_*.ply")):
    if "_light" in source:
        continue
    target = source.replace(".ply", "_light.ply")
    kept, total = compress_ply(target=target, source=source,
                               sh_degree=COMPRESS_SH_DEGREE,
                               opacity_threshold=loc["opacity_loc"],
                               scale_percentile=loc["scale_percentile"])
    before = os.path.getsize(source) / 1e6
    after = os.path.getsize(target) / 1e6
    print(f"{os.path.basename(target)}: kept {kept}/{total} blobs | "
          f"{before:.0f} -> {after:.0f} MB ({before / after:.1f}x smaller)")

if loc["floater"] == "manh":
    print("\nCòn hai bước diệt hạt ma nữa chưa tự động hoá, làm tay khi cần:")
    print("  · đếm tầm nhìn: hạt nào dưới 3 camera nhìn thấy thì bỏ "
          "(đọc pose từ sparse/0/images.bin)")
    print("  · crop tay trong SuperSplat — hai phút, và hiệu quả nhất")

## 11. Ô so sánh — SuperSplat có hiểu bộ lọc chống răng cưa không?

Ô này **không trả lời câu hỏi trên**. Nó dựng hai file để bạn tự nhìn.

Hai lượt train ngắn 7000 vòng trên cùng bộ ảnh, khác nhau **đúng một cờ**:

| lượt | cờ | file ra |
|---|---|---|
| A | không có `--antialiasing` | `<tên>_ss_aa_off_7000.ply` |
| B | có `--antialiasing` | `<tên>_ss_aa_on_7000.ply` |

Mở cả hai trong [superspl.at/editor](https://superspl.at/editor) rồi so.

**Vì sao câu hỏi này đáng hỏi.** `--antialiasing` là bộ lọc EWA của Mip
Splatting, và nó nằm **trong bộ vẽ**, không nằm trong file. File `.ply` chỉ chở
opacity và scale đã hội tụ *theo* bộ lọc đó. README của Inria nói thẳng rằng
viewer SIBR có nút bật/tắt riêng và "bạn nên bật nó khi xem cảnh train bằng
`--antialiasing`". Viewer nào không có nút tương đương thì đang vẽ mô hình bằng
một bộ lọc khác với bộ lọc nó được train — hạt sẽ trông không đúng như lúc đo.

Ô này chạy độc lập với mốc đang chọn, và ghi vào thư mục riêng, nên nó không
đụng gì tới file `.ply` chính của ô 8.

Tốn thêm khoảng hai lần 15–25 phút ở 3200px. Bỏ qua được, không ô nào sau phụ
thuộc vào nó.

In [ ]:
# ===========================================================================
# HAI LƯỢT NGẮN, KHÁC NHAU ĐÚNG MỘT CỜ
#
# Mục đích: có hai file .ply để mở cạnh nhau trong SuperSplat. Ô này không đo
# gì và không kết luận gì — bộ lọc chống răng cưa nằm trong bộ vẽ, còn viewer
# có bộ vẽ riêng của nó, nên câu trả lời phải nhìn bằng mắt trong chính viewer.
#
# Train lại từ đầu cả hai lượt, thư mục riêng, không dùng lại checkpoint nào.
# Trộn .ply train bằng bộ vẽ cũ với bộ vẽ mới đã có báo cáo hiện màn hình trống.
# ===========================================================================
import glob
import os
import shutil
import subprocess
import time

SO_SANH_ITER = 7000

# (tên ngắn, cờ thêm vào). Cả hai lượt dùng CHUNG mọi tham số còn lại của CFG,
# nên hiệu số giữa hai file đúng bằng công của một cờ.
LUOT = [("aa_off", []), ("aa_on", ["--antialiasing"])]

if not globals().get("CO_AA", True):
    raise SystemExit(
        "Bộ vẽ đang cài không có trường antialiasing, nên lượt B sẽ ra y hệt\n"
        "lượt A và so sánh này vô nghĩa. Chạy lại ô 6 với mốc V2b hoặc V2c\n"
        "để nó build nhánh có antialiasing.")

os.makedirs(OUTPUT_DIR, exist_ok=True)


def lenh_cho(thu_muc, co_them):
    """Lệnh train cho một lượt. Khác lượt kia đúng phần co_them."""
    lenh = ["python", "train.py",
            "-s", WORK,
            "-m", thu_muc,
            "--save_iterations", str(SO_SANH_ITER)]
    for co, gia_tri in CFG["cli"].items():
        # --iterations đặt lại bên dưới; mọi cờ khác giữ nguyên của cấu hình
        # chính, kể cả -r 1, --data_device cpu và --optimizer_type nếu có.
        if co != "--iterations":
            lenh += [co, str(gia_tri)]
    lenh += ["--iterations", str(SO_SANH_ITER)]
    # KHÔNG lấy CFG["co_bat"] ở đây: --antialiasing phải là khác biệt DUY NHẤT
    # giữa hai lượt, nên nó chỉ được vào từ co_them.
    lenh += co_them
    if EVAL:
        lenh.append("--eval")
    return lenh


def dong_cuoi(duong):
    """Dòng [LOG] mới nhất, hoặc dòng cuối cùng nếu chưa có mốc nào."""
    try:
        with open(duong, encoding="utf-8", errors="replace") as f:
            chu = f.read()
    except OSError:
        return ""
    dong = [d.strip() for d in chu.replace("\r", "\n").splitlines() if d.strip()]
    moc = [d for d in dong if "[LOG]" in d]
    return moc[-1] if moc else (dong[-1] if dong else "")


da_xong = []
for ten, co_them in LUOT:
    thu_muc = f"{WORK}/so_sanh_{ten}"
    log_path = f"{WORK}/so_sanh_{ten}.log"
    shutil.rmtree(thu_muc, ignore_errors=True)     # train lại từ đầu, không kế thừa

    lenh = lenh_cho(thu_muc, co_them)
    print(f"\n===== LƯỢT {ten} =====")
    print("$", " ".join(lenh), "\n")

    with open(log_path, "w") as log_file:
        p = subprocess.Popen(lenh, cwd="/content/gaussian-splatting",
                             stdout=log_file, stderr=subprocess.STDOUT, text=True)
        while p.poll() is None:
            time.sleep(30)
            tail = dong_cuoi(log_path)
            if tail:
                print(tail)

    nguon = f"{thu_muc}/point_cloud/iteration_{SO_SANH_ITER}/point_cloud.ply"
    if not os.path.exists(nguon):
        print(f"\n!! Lượt {ten} không ra file .ply. Ba mươi dòng cuối của log:\n")
        print(subprocess.run(["tail", "-30", log_path],
                             capture_output=True, text=True).stdout)
        continue

    dich = f"{OUTPUT_DIR}/{PROJECT}_ss_{ten}_{SO_SANH_ITER}.ply"
    shutil.copy(nguon, dich)
    shutil.copy(log_path, f"{OUTPUT_DIR}/{PROJECT}_ss_{ten}.log")
    da_xong.append(dich)
    print(f">>> SAVED TO DRIVE: {os.path.basename(dich)} "
          f"({os.path.getsize(dich) / 1e6:.0f} MB)")

print("\n" + "=" * 70)
if len(da_xong) == 2:
    print("Hai file đã nằm trên Drive:")
    for d in da_xong:
        print("   ", os.path.basename(d))
    print("\nMở cả hai trong https://superspl.at/editor và so bằng mắt.")
    print("Chúng khác nhau đúng một cờ --antialiasing, mọi thứ còn lại y hệt.")
    print("Ô này cố tình không kết luận hộ — bộ lọc nằm trong bộ vẽ, mà viewer")
    print("có bộ vẽ riêng của nó.")
else:
    print(f"Chỉ có {len(da_xong)}/2 lượt ra file. Đọc log ở trên.")

## 12. View the result

Download a `_light.ply` from Drive, open **https://superspl.at/editor**, and drag
the file in.

The model has no real-world scale — use the editor's scale tool to size it by eye.

Muốn nhẹ hơn nữa cho máy yếu thì đưa file qua
`npx @playcanvas/splat-transform` để xuất `.sog` — nó làm Morton ordering,
codebook k-means và nén WebP, ba việc mà viết tay lại trong notebook thì tốn
công mà file chẳng nhỏ đi byte nào.

---

### If something went wrong

**"Không tìm thấy thư mục nào chứa cả images/ và sparse/0/"** — the file you
pointed at is not a step-2 result. The photo folder from step 1 is not enough:
training needs `sparse/0/*.bin`, which only step 2 produces. Ô 4 prints what
your file actually contained, which usually makes it obvious.

**Uploading took forever** — you probably zipped `distorted/database.db` along
with everything else. It is several gigabytes and training never opens it. Leave
it out next time:
`zip -r -0 Meo.zip Meo_3d -x "Meo_3d/distorted/*" "Meo_3d/stereo/*"`

**Hết RAM ở ô 4 hoặc ô 8** — RAM hệ thống mới là nút thắt, không phải VRAM.
Ô 5 tính trước và dừng lại khi vượt trần 10,5 GB; nó nói luôn cỡ ảnh hiện tại
thì chứa được tối đa bao nhiêu tấm. Bớt ảnh, hoặc quay lại chặng 0 thu nhỏ
ảnh rồi chạy lại chặng 1 và 2 — nâng độ phân giải chỉ ở chặng 3 là vô nghĩa,
nó bị khoá từ chặng 0.

**Hết VRAM giữa lúc train** — đọc mốc `[LOG]` cuối cùng trong `train.log`:
nó in số hạt và VRAM đỉnh thật. Hạ `MAX_GAUSSIANS` trong bảng preset ở ô 6
đi 25% rồi chạy lại từ ô 7. **Đừng** quay về cách cũ là nâng
`densify_grad_threshold` — đó chính là thứ làm chữ nhỏ mờ đi.

**Chữ nhỏ vẫn mờ dù đã chạy V2** — bảng preset chỉ giảm nhẹ triệu chứng chứ
không chữa được gốc. Gốc là *gradient collision*: một hạt to trùm lên vùng chữ
thì gradient các phía ngược dấu nhau, cộng lại gần bằng 0, nên hạt đó không bao
giờ đạt ngưỡng để bị tách — hạ ngưỡng xuống bao nhiêu cũng vô ích với một hạt
có điểm số bằng 0. Chữa đúng gốc là lấy trị tuyệt đối từng gradient pixel rồi
mới cộng (AbsGS), và việc đó phải build lại rasterizer. Lưu ý ngược trực giác:
dùng AbsGS thì phải **nâng** `densify_grad_threshold` lên 0.0004–0.0008, không
phải hạ, vì tín hiệu mạnh hơn hẳn.

**Build `3dgs_accel` hỏng ở ô cài đặt** — ô đó không sập: nó lùi về `dr_aa`, giữ
`--antialiasing` và gỡ `--optimizer_type` khỏi cấu hình. Dòng
`bộ vẽ : ... | SparseGaussianAdam False` ở cuối ô cài đặt nói cho bạn biết điều đó đã
xảy ra. Mốc chạy được lúc đó là V2b chứ không phải V2c, nên đừng ghi kết quả vào
ô V2c của bảng đo.

**Mô hình mở trong SuperSplat trông khác lúc render ở ô 9** — `--antialiasing`
là bộ lọc nằm **trong bộ vẽ**, không nằm trong file `.ply`. File chỉ chở opacity
và scale đã hội tụ theo bộ lọc đó. README của Inria nói rõ là viewer SIBR có nút
bật/tắt riêng và phải bật nó khi xem cảnh train bằng `--antialiasing`. Ô 11 dựng
sẵn hai file khác nhau đúng một cờ để bạn tự so trong SuperSplat.

**Màn hình trống khi mở file cũ sau khi đổi bộ vẽ** — đừng dùng lại `.ply` train
từ bộ vẽ trước. Train lại từ đầu; các ô ở đây không dùng lại checkpoint nào.

**The light copy looks flat, reflections gone** — set `COMPRESS_SH_DEGREE = 3`
và chạy lại ô 10. The full file is still on Drive; no need to retrain.

**Colab disconnected halfway** — run again from ô 1. Checkpoints already
copied to Drive are still good.

**Only a few images were reconstructed in step 2** — ô 4 tells you this
before training starts (`Đã định vị : 90/320 ảnh`). The problem is the photos,
not the training. Neighbouring shots need roughly 60-80% overlap.